# GPU Profiling for xPU-Simulator Calibration

Profiles MATMUL and vector ops on the current GPU using `torch.cuda.Event` for precise timing.
Results are used to calibrate the xPU-simulator's GPU cost model.

**Instructions:** Runtime → Change runtime type → select GPU (A100 preferred). Then Run All.

In [ ]:
import torch
import torch.nn.functional as F
import csv
import io
import sys

assert torch.cuda.is_available(), "No CUDA device — change runtime to GPU"

props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name
SM_COUNT = props.multi_processor_count
MEM_GB = round(props.total_mem / 1024**3, 1)
COMPUTE_CAP = f"{props.major}.{props.minor}"

print(f"GPU: {GPU_NAME}")
print(f"SMs: {SM_COUNT}  Memory: {MEM_GB} GB  Compute: {COMPUTE_CAP}")
print(f"CUDA: {torch.version.cuda}  Torch: {torch.__version__}")

In [ ]:
# --- Config ---
WARMUP = 20
ITERS = 100
DTYPE = torch.float16
DEVICE = "cuda"

# Known GPU specs for efficiency calculation
GPU_SPECS = {
    "A100": {"tflops": 312, "bw_GBs": 2039},
    "H100": {"tflops": 989, "bw_GBs": 3350},
    "T4":   {"tflops": 65,  "bw_GBs": 320},
    "L4":   {"tflops": 121, "bw_GBs": 300},
    "V100": {"tflops": 125, "bw_GBs": 900},
}

SPEC = None
for key, val in GPU_SPECS.items():
    if key in GPU_NAME:
        SPEC = val
        break
if SPEC:
    print(f"Matched spec: {SPEC['tflops']} TFLOPS, {SPEC['bw_GBs']} GB/s")
else:
    print(f"Unknown GPU '{GPU_NAME}' — will still profile, but can't derive efficiency factors")

In [ ]:
def bench(fn, warmup=WARMUP, iters=ITERS):
    """Return median latency in microseconds."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record()
        torch.cuda.synchronize()
        times.append(s.elapsed_time(e) * 1000)  # ms → us
    times.sort()
    return times[len(times) // 2]

## 1. MATMUL Profiling

In [ ]:
MATMUL_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 14336),
    (1, 14336, 4096),
    (128, 4096, 4096),
    (1024, 1024, 1024),
    (1024, 4096, 4096),
    (1024, 4096, 14336),
    (1024, 14336, 4096),
    (1024, 4096, 11008),
    (1024, 11008, 4096),
    (1024, 8192, 8192),
    (1024, 8192, 28672),
    (2048, 4096, 4096),
    (4096, 4096, 4096),
]

matmul_results = []
print(f"{'Shape':<22} {'Latency (us)':>14} {'TFLOPS':>10}")
print("-" * 50)

for M, K, N in MATMUL_SHAPES:
    A = torch.randn(M, K, dtype=DTYPE, device=DEVICE)
    B = torch.randn(K, N, dtype=DTYPE, device=DEVICE)
    lat = bench(lambda: torch.mm(A, B))
    flops = 2 * M * K * N
    tflops = flops / lat / 1e6
    mem = (M*K + K*N + M*N) * 2
    r = {"op": "MATMUL", "shape": f"{M}x{K}x{N}",
         "latency_us": round(lat, 2), "flops": flops,
         "tflops": round(tflops, 2), "mem_bytes": mem,
         "arith_intensity": round(flops / mem, 1)}
    matmul_results.append(r)
    print(f"{r['shape']:<22} {lat:>14.2f} {tflops:>10.2f}")
    del A, B
    torch.cuda.empty_cache()

peak_tflops = max(r["tflops"] for r in matmul_results)
print(f"\nPeak MATMUL: {peak_tflops:.2f} TFLOPS")
if SPEC:
    print(f"Efficiency: {peak_tflops / SPEC['tflops']:.3f} ({peak_tflops:.1f} / {SPEC['tflops']} TFLOPS)")

## 2. Elementwise Ops (memory-bound baseline)

In [ ]:
VECTOR_SHAPES = [
    (1, 4096),
    (128, 4096),
    (1024, 4096),
    (1024, 8192),
    (1024, 14336),
    (2048, 4096),
    (4096, 4096),
]

def profile_vec(op_name, make_fn, run_fn, flops_fn, mem_fn):
    """Generic vector op profiler."""
    results = []
    print(f"\n--- {op_name} ---")
    print(f"{'Shape':<16} {'Latency (us)':>14} {'BW (GB/s)':>12}")
    print("-" * 46)
    for rows, cols in VECTOR_SHAPES:
        tensors = make_fn(rows, cols)
        lat = bench(lambda: run_fn(*tensors))
        flops = flops_fn(rows, cols)
        mem = mem_fn(rows, cols)
        bw = mem / lat * 1e6 / 1e9 if lat > 0 else 0
        r = {"op": op_name, "shape": f"{rows}x{cols}",
             "latency_us": round(lat, 2), "flops": flops,
             "tflops": round(flops / lat / 1e6, 4) if lat > 0 else 0,
             "mem_bytes": mem,
             "arith_intensity": round(flops / mem, 1) if mem > 0 else 0}
        results.append(r)
        print(f"{r['shape']:<16} {lat:>14.2f} {bw:>12.1f}")
        for t in tensors:
            del t
        torch.cuda.empty_cache()
    return results

In [ ]:
add_results = profile_vec(
    "ADD",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),
                  torch.randn(r, c, dtype=DTYPE, device=DEVICE)),
    lambda a, b: torch.add(a, b),
    lambda r, c: r * c,
    lambda r, c: (2 * r * c + r * c) * 2,  # 2 in + 1 out, fp16
)

In [ ]:
mul_results = profile_vec(
    "MUL",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),
                  torch.randn(r, c, dtype=DTYPE, device=DEVICE)),
    lambda a, b: torch.mul(a, b),
    lambda r, c: r * c,
    lambda r, c: (2 * r * c + r * c) * 2,
)

In [ ]:
relu_results = profile_vec(
    "RELU",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),),
    lambda x: F.relu(x),
    lambda r, c: r * c,
    lambda r, c: (r * c + r * c) * 2,  # 1 in + 1 out
)

## 3. Activation Functions

In [ ]:
silu_results = profile_vec(
    "SILU",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),),
    lambda x: F.silu(x),
    lambda r, c: 5 * r * c,
    lambda r, c: (r * c + r * c) * 2,
)

In [ ]:
gelu_results = profile_vec(
    "GELU",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),),
    lambda x: F.gelu(x),
    lambda r, c: 8 * r * c,
    lambda r, c: (r * c + r * c) * 2,
)

## 4. Reduction Ops (multi-pass)

In [ ]:
ln_results = profile_vec(
    "LAYERNORM",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),
                  torch.ones(c, dtype=DTYPE, device=DEVICE),
                  torch.zeros(c, dtype=DTYPE, device=DEVICE)),
    lambda x, w, b: F.layer_norm(x, (x.shape[-1],), w, b),
    lambda r, c: 7 * r * c,
    lambda r, c: (r * c + r * c) * 2,
)

In [ ]:
sm_results = profile_vec(
    "SOFTMAX",
    lambda r, c: (torch.randn(r, c, dtype=DTYPE, device=DEVICE),),
    lambda x: F.softmax(x, dim=-1),
    lambda r, c: 5 * r * c,
    lambda r, c: (r * c + r * c) * 2,
)

## 5. RoPE (rotary position embedding)

In [ ]:
rope_results = []
print("--- ROPE ---")
print(f"{'Shape':<16} {'Latency (us)':>14} {'BW (GB/s)':>12}")
print("-" * 46)

for rows, cols in VECTOR_SHAPES:
    half = cols // 2
    x = torch.randn(rows, cols, dtype=DTYPE, device=DEVICE)
    cos_c = torch.randn(rows, half, dtype=DTYPE, device=DEVICE)
    sin_c = torch.randn(rows, half, dtype=DTYPE, device=DEVICE)

    def rope_fn(x=x, cos_c=cos_c, sin_c=sin_c, half=half):
        x0, x1 = x[..., :half], x[..., half:]
        y0 = x0 * cos_c - x1 * sin_c
        y1 = x0 * sin_c + x1 * cos_c
        return torch.cat([y0, y1], dim=-1)

    lat = bench(rope_fn)
    flops = 6 * rows * cols
    mem = (rows * cols + 2 * rows * half + rows * cols) * 2
    bw = mem / lat * 1e6 / 1e9 if lat > 0 else 0
    r = {"op": "ROPE", "shape": f"{rows}x{cols}",
         "latency_us": round(lat, 2), "flops": flops,
         "tflops": round(flops / lat / 1e6, 4) if lat > 0 else 0,
         "mem_bytes": mem,
         "arith_intensity": round(flops / mem, 1) if mem > 0 else 0}
    rope_results.append(r)
    print(f"{r['shape']:<16} {lat:>14.2f} {bw:>12.1f}")
    del x, cos_c, sin_c
    torch.cuda.empty_cache()

## 6. Results & Calibration

In [ ]:
all_results = (matmul_results + add_results + mul_results + relu_results +
               silu_results + gelu_results + ln_results + sm_results + rope_results)

# --- Derive efficiency factors ---
print("=" * 60)
print("DERIVED EFFICIENCY FACTORS")
print("=" * 60)

peak_mm = max(r["tflops"] for r in matmul_results)
print(f"\nPeak MATMUL throughput: {peak_mm:.2f} TFLOPS")

# Memory BW from simple ops (ADD/MUL/RELU are pure memory-bound)
membw_results = add_results + mul_results + relu_results
bws = [r["mem_bytes"] / r["latency_us"] * 1e6 / 1e9
       for r in membw_results if r["latency_us"] > 5]  # skip tiny/noisy
peak_bw = max(bws) if bws else 0
avg_bw = sum(bws) / len(bws) if bws else 0
print(f"Effective mem BW: peak {peak_bw:.1f} GB/s, avg {avg_bw:.1f} GB/s")

if SPEC:
    mm_eff = peak_mm / SPEC["tflops"]
    mem_eff = peak_bw / SPEC["bw_GBs"]
    print(f"\n  matmul_fp16:  {mm_eff:.3f}  (current model: 0.70)")
    print(f"  memory:       {mem_eff:.3f}  (current model: 0.80)")

# Static overhead from tiny matmuls
tiny = [r for r in matmul_results if r["shape"].startswith("1x")]
if tiny:
    min_lat = min(r["latency_us"] for r in tiny)
    print(f"  static overhead (tiny matmul): ~{min_lat:.1f} us  (current model: 5.0 us)")

# Bandwidth ratio for multi-pass ops vs single-pass
print("\n--- Effective memory passes (vs ADD baseline) ---")
add_bws = {r["shape"]: r["mem_bytes"] / r["latency_us"] * 1e6 / 1e9
           for r in add_results if r["latency_us"] > 5}

for op_name, op_results in [("SILU", silu_results), ("GELU", gelu_results),
                             ("LAYERNORM", ln_results), ("SOFTMAX", sm_results),
                             ("ROPE", rope_results)]:
    ratios = []
    for r in op_results:
        if r["shape"] in add_bws and r["latency_us"] > 5:
            # Same data size, longer time → more memory passes
            op_bw = r["mem_bytes"] / r["latency_us"] * 1e6 / 1e9
            ratio = add_bws[r["shape"]] / op_bw  # >1 means slower → more passes
            ratios.append(ratio)
    if ratios:
        avg_ratio = sum(ratios) / len(ratios)
        print(f"  {op_name:<12} {avg_ratio:.2f}x vs ADD")

In [ ]:
# --- CSV output ---
output = io.StringIO()
writer = csv.DictWriter(output, fieldnames=[
    "op", "shape", "latency_us", "flops", "tflops", "mem_bytes", "arith_intensity"])
writer.writeheader()
for r in all_results:
    writer.writerow(r)

csv_text = output.getvalue()

fname = f"gpu_profiling_{GPU_NAME.replace(' ', '_')}.csv"
with open(fname, "w") as f:
    f.write(f"# GPU: {GPU_NAME}\n")
    f.write(f"# SMs: {SM_COUNT}  Memory: {MEM_GB}GB  Compute: {COMPUTE_CAP}\n")
    f.write(f"# CUDA: {torch.version.cuda}  Torch: {torch.__version__}\n")
    f.write(f"# Warmup: {WARMUP}  Iters: {ITERS}  Dtype: fp16\n")
    f.write(csv_text)

print(f"Saved to: {fname}")
print(f"\n--- Copy CSV below to import into xPU-simulator ---\n")
print(csv_text)

In [ ]:
# Download the CSV file (Colab only)
try:
    from google.colab import files
    files.download(fname)
except ImportError:
    print("Not running in Colab — file saved locally")